# The Competition Discount — Step 2: Inspect and clean
Loads the raw file from Step 1, inspects duplicates, airline codes and impossible values, then cleans the table, adds derived fields and saves the clean CSV that is loaded into Supabase.


In [ ]:
# 1. Load the raw file saved in Step 1 (no re-download)
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

PROJECT_DIR = '/content/drive/MyDrive/the-competition-discount'
RAW_PATH = f'{PROJECT_DIR}/data/raw/table6_2019_onward_raw.csv'

raw = pd.read_csv(RAW_PATH, dtype=str)
df = raw.drop(columns=[c for c in raw.columns if c.startswith('location_')])   # empty map fields

num_cols = ['year', 'quarter', 'nsmiles', 'passengers', 'fare', 'large_ms', 'fare_lg', 'lf_ms', 'fare_low']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print(f'{len(df):,} rows, {df.shape[1]} columns:', list(df.columns))

In [ ]:
# 2. The duplicate route-quarter rows: what are they?
key = ['year', 'quarter', 'citymarketid_1', 'citymarketid_2']
dups = df[df.duplicated(key, keep=False)].sort_values(key)
print(f'{len(dups)} rows involved')
dups[key + ['city1', 'city2', 'nsmiles', 'passengers', 'fare', 'carrier_lg', 'carrier_low', 'tbl', 'table_1_flag', 'tbl6pk']]

In [ ]:
# 3. Which airline codes appear? (needed to build the low-cost list from real codes)
lg = df['carrier_lg'].value_counts().rename('as_dominant')
low = df['carrier_low'].value_counts().rename('as_cheapest')
codes = pd.concat([lg, low], axis=1).fillna(0).astype(int).sort_values('as_dominant', ascending=False)
print(f'{len(codes)} distinct codes')
print(codes.to_string())

In [ ]:
# 4. Impossible or suspicious values
checks = {
    'fare <= 0':                    (df['fare'] <= 0).sum(),
    'fare_lg <= 0':                 (df['fare_lg'] <= 0).sum(),
    'fare_low <= 0':                (df['fare_low'] <= 0).sum(),
    'nsmiles <= 0':                 (df['nsmiles'] <= 0).sum(),
    'passengers <= 0':              (df['passengers'] <= 0).sum(),
    'large_ms outside 0-1':         ((df['large_ms'] < 0) | (df['large_ms'] > 1)).sum(),
    'lf_ms outside 0-1':            ((df['lf_ms'] < 0) | (df['lf_ms'] > 1)).sum(),
    'fare_low > fare_lg':           (df['fare_low'] > df['fare_lg']).sum(),
    'same airline dominant+cheapest': (df['carrier_lg'] == df['carrier_low']).sum(),
    'any numeric failed to convert':  df[num_cols].isna().any(axis=1).sum(),
}
for k, v in checks.items():
    print(f'{k:<34}{v:>8,}')

print('\nSummary of the key numbers:')
print(df[['nsmiles', 'passengers', 'fare', 'large_ms', 'fare_lg', 'lf_ms', 'fare_low']].describe().round(2).to_string())

In [ ]:
# 5. Remove the 2 broken duplicate rows
# A valid tbl6pk = year + quarter + citymarketid_1 + citymarketid_2 (e.g. 2019 1 34986 35096 -> 201913498635096)
expected_pk = df['year'].astype(int).astype(str) + df['quarter'].astype(int).astype(str) + df['citymarketid_1'] + df['citymarketid_2']
bad_pk = df['tbl6pk'] != expected_pk
print('Rows with a malformed tbl6pk:', bad_pk.sum())
print(df.loc[bad_pk, ['year', 'quarter', 'city1', 'city2', 'tbl6pk']].to_string())

df = df.loc[~bad_pk].copy()
key = ['year', 'quarter', 'citymarketid_1', 'citymarketid_2']
print(f'\nRows left: {len(df):,} | duplicate route-quarters left: {df.duplicated(key).sum()}')


In [ ]:
# 6. Airline names and groups
carrier_names = {
    'AA': 'American Airlines', 'DL': 'Delta Air Lines', 'UA': 'United Airlines', 'AS': 'Alaska Airlines',
    'HA': 'Hawaiian Airlines', 'WN': 'Southwest Airlines', 'G4': 'Allegiant Air', 'F9': 'Frontier Airlines',
    'NK': 'Spirit Airlines', 'MX': 'Breeze Airways', 'SY': 'Sun Country Airlines', 'XP': 'Avelo Airlines',
    'B6': 'JetBlue Airways', 'KG': 'Key Lime Air (Denver Air Connection)',
    # small regional carriers, named from their IATA codes; grouped as regional_other, so they sit outside the low-cost vs full-service comparisons
    '3M': 'Silver Airways', '9K': 'Cape Air', '9X': 'Southern Airways Express', 'M5': 'Kenmore Air',
    '4B': 'Boutique Air', 'AN': 'Advanced Air', 'LF': 'Contour Airlines', 'TJ': 'Tradewind Aviation',
    'NE': 'Unidentified regional carrier',
}
LOW_COST = {'WN', 'G4', 'F9', 'NK', 'MX', 'SY', 'XP'}
HYBRID = {'B6'}
FULL_SERVICE = {'AA', 'DL', 'UA', 'AS', 'HA'}

def group(code):
    if code in LOW_COST: return 'low_cost'
    if code in HYBRID: return 'hybrid'
    if code in FULL_SERVICE: return 'full_service'
    return 'regional_other'

df['carrier_lg_name'] = df['carrier_lg'].map(carrier_names)
df['carrier_low_name'] = df['carrier_low'].map(carrier_names)
df['lg_group'] = df['carrier_lg'].map(group)
df['low_group'] = df['carrier_low'].map(group)
df['lcc_present'] = df['low_group'] == 'low_cost'                                   # main run
df['lcc_present_incl_b6'] = df['low_group'].isin(['low_cost', 'hybrid'])            # sensitivity check

unnamed = set(df['carrier_lg']).union(df['carrier_low']) - set(carrier_names)
print('Codes without a name:', unnamed or 'none')
print('\nDominant airline group:'); print(df['lg_group'].value_counts().to_string())
print('\nCheapest airline group:'); print(df['low_group'].value_counts().to_string())


In [ ]:
# 7. Derived fields: period, distance band, COVID flag, yield, revenue estimates
df['year'] = df['year'].astype(int)
df['quarter'] = df['quarter'].astype(int)
df['period'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)
df['period_index'] = df['year'] * 4 + df['quarter'] - 1          # consecutive integers: makes "previous quarter" checks easy in SQL
df['route_id'] = df['citymarketid_1'] + '-' + df['citymarketid_2']

df['distance_band'] = pd.cut(df['nsmiles'], bins=[0, 500, 1000, 1500, float('inf')],
                             labels=['0-500', '500-1000', '1000-1500', '1500+']).astype(str)
df['covid_flag'] = df['year'].isin([2020, 2021])

per = pd.PeriodIndex(df['period'], freq='Q')
df['days_in_quarter'] = ((per.end_time.normalize() - per.start_time).days + 1).astype(int)

df['fare_per_mile'] = (df['fare'] / df['nsmiles']).round(4)
df['est_route_rev_q'] = (df['passengers'] * df['days_in_quarter'] * df['fare']).round(0)
df['est_dominant_rev_q'] = (df['passengers'] * df['large_ms'] * df['days_in_quarter'] * df['fare_lg']).round(0)

print('Routes per distance band:'); print(df['distance_band'].value_counts().sort_index().to_string())
print('\nDays in quarter seen:', sorted(df['days_in_quarter'].unique()))
print('COVID-flagged rows:', f"{df['covid_flag'].sum():,}")


In [ ]:
# 8. Data-quality log
dq_log = [
    ('Download', '167,886 rows, 2019Q1-2026Q1; matches the server count by year exactly'),
    ('Reconciliation', 'Route counts match DOT published reports for 2020Q1 (5,412), 2023Q2 (6,084), 2025Q1 (6,048)'),
    ('Report note', 'DOT Q4 2025 report text says 6,611 markets for Q3 2025; data shows 6,611 in 2024Q3 and 7,019 in 2025Q3 (likely a year typo in the report)'),
    ('Dropped columns', '10 location_* columns: map fields, empty or almost empty'),
    ('Duplicates', f'{bad_pk.sum()} rows removed: exact copies of 2019Q1 routes with malformed tbl6pk keys'),
    ('Impossible values', 'None: no zero/negative fares, distances or passengers; shares within 0-1; fare_low never above fare_lg'),
    ('Dominant = cheapest', f"{(df['carrier_lg'] == df['carrier_low']).sum():,} rows ({(df['carrier_lg'] == df['carrier_low']).mean():.0%}): no airline undercuts the leader; a pricier low-cost airline would be invisible"),
    ('Passenger skew', 'Median 35 passengers/day vs mean 191; report route-count and passenger-weighted views'),
    ('COVID', '2020-2021 flagged; route counts fell from 6,405 (2019Q2) to 2,102 (2020Q2)'),
    ('Low-cost list', 'WN, G4, F9, NK, MX, SY, XP; JetBlue (B6) = hybrid, low-cost only in the sensitivity check'),
    ('Revenue estimates', 'passengers/day x days x fare; fares include taxes, fees and first class, so estimates run high'),
]
dq = pd.DataFrame(dq_log, columns=['check', 'result'])
print(dq.to_string(index=False))


In [ ]:
# 9. Save the clean table: only the columns Supabase needs
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

keep = ['route_id', 'period', 'year', 'quarter', 'period_index', 'citymarketid_1', 'citymarketid_2', 'city1', 'city2',
        'nsmiles', 'distance_band', 'passengers', 'fare', 'fare_per_mile',
        'carrier_lg', 'carrier_lg_name', 'lg_group', 'large_ms', 'fare_lg',
        'carrier_low', 'carrier_low_name', 'low_group', 'lf_ms', 'fare_low',
        'lcc_present', 'lcc_present_incl_b6', 'covid_flag', 'days_in_quarter', 'est_route_rev_q', 'est_dominant_rev_q']
clean = df[keep].sort_values(['route_id', 'period_index'])
clean.to_csv(f'{PROCESSED_DIR}/route_quarters_clean.csv', index=False)
dq.to_csv(f'{PROCESSED_DIR}/data_quality_log.csv', index=False)

print(f'Saved {len(clean):,} rows x {clean.shape[1]} columns to {PROCESSED_DIR}/route_quarters_clean.csv')
print('Distinct routes:', clean['route_id'].nunique())
clean.head(3).T
